In [9]:
import pandas as pd

df = pd.read_csv('../data/raw/clean_jobs.csv')
print(df.shape)
print(df.columns.tolist())
df['title'].value_counts().head(20)

(1048, 10)
['id', 'title', 'company', 'location', 'link', 'source', 'date_posted', 'work_type', 'employment_type', 'description']


title
Data Analyst                         152
Data Scientist                        75
Data Engineer                         58
Data Scientist, Product Analytics     29
Machine Learning Engineer             21
Data Engineer, Product Analytics      20
Senior Data Engineer                  13
Senior Data Analyst                   12
Junior Data Scientist                 11
AI/ML Engineer                         9
**** *******                           9
Junior Data Analyst                    8
Data Analyst II                        7
Business Data Analyst                  7
**** ********, *********               7
Data Analyst I                         6
**** *********                         6
**** ********                          6
Data Products Analyst, YouTube         5
Data Engineer I                        5
Name: count, dtype: int64

In [10]:
# Filter to only Data Analyst and Data Scientist roles (our defined project scope)
mask = df['title'].str.contains('Data Analyst|Data Scientist', case=False, na=False)
filtered_df = df[mask].copy()

print("Filtered shape:", filtered_df.shape)
filtered_df['title'].value_counts()

Filtered shape: (481, 10)


title
Data Analyst                                                         152
Data Scientist                                                        75
Data Scientist, Product Analytics                                     29
Senior Data Analyst                                                   12
Junior Data Scientist                                                 11
                                                                    ... 
Data Scientist, Quantitative Research                                  1
Data Scientist (L5) - Ads (Forecasting)                                1
Data Scientist (Python) - Mumbai - Assistant Vice President            1
Vice President Data Scientist Marketing Analytics - Consumer Bank      1
Lead Data Scientist - Finance Technology                               1
Name: count, Length: 160, dtype: int64

In [11]:
filtered_df.to_csv('../data/processed/da_ds_filtered_jobs.csv', index=False)
print("Saved successfully")

Saved successfully


In [12]:
import os
print(os.path.exists('../data/processed/da_ds_filtered_jobs.csv'))

True


# Data Cleaning and EDA

In [13]:
filtered_df.isnull().sum()  # Check for missing Values

id                   0
title                0
company              0
location             6
link                 0
source              26
date_posted         13
work_type          481
employment_type    481
description          2
dtype: int64

In [14]:
# Drop columns that are entirely empty
clean_df = filtered_df.drop(columns=['work_type', 'employment_type'])

# Drop rows with missing description (critical field, can't impute meaningfully)
clean_df = clean_df.dropna(subset=['description'])

# Fill minor missing values with 'Unknown' instead of dropping (preserves more data)
clean_df['location'] = clean_df['location'].fillna('Unknown')
clean_df['source'] = clean_df['source'].fillna('Unknown')

print("Shape after cleaning:", clean_df.shape)
clean_df.isnull().sum()

Shape after cleaning: (479, 8)


id              0
title           0
company         0
location        0
link            0
source          0
date_posted    13
description     0
dtype: int64

In [16]:
clean_df[clean_df['date_posted'].isnull()][['title', 'company', 'date_posted']] # Actual Rows shows missing data_posted

,title,company,date_posted
946,Data Analyst (100% Remote),Lensa,NaN
948,Data Analyst,Sharaf DG,NaN
949,Data Analyst,Itaú Chile,NaN
950,"Data Analyst II, SMART",Wayfair,NaN
955,Senior Data Analyst,Government of Alberta,NaN
960,Data Analyst V,Texas Health and Human Services,NaN
964,Data Scientist,The Value Maximizer,NaN
965,Data Scientist (L5) - Member Algorithm Foundat...,Netflix,NaN
966,Data Scientist,Cadent,NaN
975,"Data Scientist, Quantitative Research",ICE,NaN


In [17]:
clean_df['date_posted'] = clean_df['date_posted'].fillna('Unknown')
# filling the missing rows with unknown

In [21]:
clean_df.isnull().sum()

id             0
title          0
company        0
location       0
link           0
source         0
date_posted    0
description    0
dtype: int64

In [22]:
print("Exact duplicate rows:", clean_df.duplicated().sum())
print("Duplicate by title+company:", clean_df.duplicated(subset=['title', 'company']).sum())

Exact duplicate rows: 0
Duplicate by title+company: 83


In [23]:
print("Duplicate by title+company+location:", clean_df.duplicated(subset=['title', 'company', 'location']).sum())

Duplicate by title+company+location: 26


In [24]:
clean_df = clean_df.drop_duplicates(subset=['title', 'company', 'location'])
print("Shape after removing duplicates:", clean_df.shape)

Shape after removing duplicates: (453, 8)


In [25]:
masked = clean_df[clean_df['title'].str.contains(r'\*', regex=True, na=False)]
print("Masked/corrupted titles:", len(masked))
masked[['title', 'company']]

Masked/corrupted titles: 0


,title,company


In [26]:
clean_df.to_csv('../data/processed/da_ds_cleaned.csv', index=False)
print("Saved successfully. Final shape:", clean_df.shape)

Saved successfully. Final shape: (453, 8)
